# NeuralStock: Deep Learning for E-commerce Inventory Demand Forecasting

**Domain:** E-Commerce · **Skills:** Deep Learning (LSTM/MLP), Time Series, Streamlit, Cloud Deployment

Structure: **Data Generation → EDA → Preprocessing → Feature Engineering → Modelling → Evaluation → Insights**. Run every cell top-to-bottom after `pip install -r requirements.txt`.

## 1. Data Generation

The primary dataset (`data/ecommerce_inventory_demand.csv`) is provided. `data_generator.py` can also produce a synthetic dataset matching the same schema, useful for testing the pipeline independently.

In [ ]:
%matplotlib inline

import sys
sys.path.insert(0, '../scripts')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

from preprocess import load_raw, handle_missing_values, TARGET

raw_df = load_raw('../data/ecommerce_inventory_demand.csv')
print('Rows:', len(raw_df), '| Products:', raw_df['product_id'].nunique())
raw_df.head()

## 2. Exploratory Data Analysis

In [ ]:
df = handle_missing_values(raw_df)

print('Missing values (raw):')
print(raw_df.isnull().sum())
print()
print(df[[TARGET, 'unit_price', 'stock_on_hand', 'discount_pct']].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(df[TARGET], bins=40, kde=True, ax=ax)
ax.set_title('Distribution of Daily Units Sold')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
cat_avg = df.groupby('product_category')[TARGET].mean().sort_values(ascending=False)
sns.barplot(x=cat_avg.index, y=cat_avg.values, ax=ax)
ax.set_title('Average Units Sold by Product Category')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
monthly = df.groupby('month')[TARGET].mean()
sns.lineplot(x=monthly.index, y=monthly.values, marker='o', ax=ax)
ax.set_title('Average Units Sold by Month (Seasonality)')
ax.set_xticks(range(1,13))
plt.show()

### Autocorrelation (ACF)
Computed manually below so the notebook has no hard dependency on `statsmodels`. If you have it installed, `statsmodels.graphics.tsaplots.plot_acf` / `plot_pacf` give the same information plus PACF.

In [ ]:
def manual_acf(series, nlags=20):
    series = series - series.mean()
    denom = np.sum(series ** 2)
    return [1.0] + [np.sum(series[l:] * series[:-l]) / denom for l in range(1, nlags+1)]

top_product = df['product_id'].value_counts().idxmax()
series = df[df['product_id'] == top_product].sort_values('date')[TARGET].reset_index(drop=True)
acf_vals = manual_acf(series.to_numpy(), nlags=20)

fig, ax = plt.subplots(figsize=(7,4))
ax.stem(range(len(acf_vals)), acf_vals)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'ACF of units_sold — {top_product}')
plt.show()

**Note:** Each SKU is sampled irregularly (~1 record every 6 days) rather than daily, so lag/rolling features below operate in *observation order* per SKU rather than strict calendar days — documented as a limitation in the project report.

## 3. Preprocessing

In [ ]:
from preprocess import remove_price_outliers, add_calendar_features

df_clean = remove_price_outliers(df)
df_clean = add_calendar_features(df_clean)
df_clean[['unit_price']].describe()

## 4. Feature Engineering

In [ ]:
from preprocess import add_lag_and_rolling_features, encode_categoricals

df_feat = add_lag_and_rolling_features(df_clean)
df_feat = encode_categoricals(df_feat)
df_feat = df_feat.dropna(subset=['roll_mean_30','roll_std_30','lag_14']).reset_index(drop=True)
print('Feature-engineered shape:', df_feat.shape)
df_feat.filter(regex='lag_|roll_|category_').head()

In [ ]:
from preprocess import chronological_split, fit_scaler, apply_scaler, NUMERIC_FEATURES

train_df, test_df = chronological_split(df_feat, test_size=0.2)
scaler = fit_scaler(train_df)
train_scaled = apply_scaler(train_df, scaler)
test_scaled = apply_scaler(test_df, scaler)
print('Train:', train_scaled.shape, ' Test:', test_scaled.shape)

## 5. Modelling

Trains the real `DemandLSTM` and `DemandMLP` from `model.py`. Requires PyTorch (`pip install -r requirements.txt`).

In [ ]:
import torch
from torch.utils.data import DataLoader
from model import DemandLSTM, DemandMLP, SlidingWindowDataset, TabularDataset, set_seeds
from preprocess import bridge_train_test_for_windows

set_seeds(42)
SEQUENCE_LENGTH = 14

# Without bridging, every product's first SEQUENCE_LENGTH test rows have no
# prior TEST history to build a window from and get silently dropped -- on
# this dataset that would throw away ~75% of the test set. Prepending each
# product's trailing TRAINING rows (never future data) recovers a window
# for every test row and makes the LSTM's test set match the MLP's.
test_bridged = bridge_train_test_for_windows(train_scaled, test_scaled, sequence_length=SEQUENCE_LENGTH)

# predict_residual=True: the LSTM is trained to predict
# (units_sold - roll_mean_7) rather than raw units_sold. Training on the
# raw value lets the model minimize loss almost entirely by learning a
# constant near the training mean, since day-to-day variation is small
# relative to the overall level -- see model.py's SlidingWindowDataset
# docstring for the full explanation.
train_seq_ds = SlidingWindowDataset(train_scaled, NUMERIC_FEATURES, sequence_length=SEQUENCE_LENGTH, predict_residual=True)
test_seq_ds = SlidingWindowDataset(test_bridged, NUMERIC_FEATURES, sequence_length=SEQUENCE_LENGTH, predict_residual=True)
train_loader = DataLoader(train_seq_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_seq_ds, batch_size=32, shuffle=False)

print('LSTM training windows:', len(train_seq_ds), '| test windows:', len(test_seq_ds), '(raw test rows:', len(test_scaled), ')')

In [ ]:
# Full training loop with TensorBoard logging lives in train.py — reuse it here
# so the notebook and the script stay in sync rather than duplicating logic.
sys.path.insert(0, '../scripts')
from train import train_one_model, evaluate, naive_baseline_metrics
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir='../runs/notebook_run')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

lstm_model = DemandLSTM(input_size=len(NUMERIC_FEATURES))
lstm_model = train_one_model(lstm_model, train_loader, test_loader,
                              epochs=40, lr=1e-3, writer=writer, tag='LSTM', device=device)
# predict_residual=True adds the roll_mean_7 baseline back onto both the
# model's output and the target before scoring, so these metrics are in
# real units_sold units even though the model itself predicts a residual.
lstm_metrics = evaluate(lstm_model, test_loader, device, predict_residual=True)
lstm_naive = naive_baseline_metrics(test_loader, predict_residual=True)
print({k: v for k, v in lstm_metrics.items() if k not in ('y_true', 'y_pred')})
print('Naive (predict = roll_mean_7):', lstm_naive)

In [ ]:
train_tab_ds = TabularDataset(train_scaled, NUMERIC_FEATURES)
test_tab_ds = TabularDataset(test_scaled, NUMERIC_FEATURES)
train_tab_loader = DataLoader(train_tab_ds, batch_size=32, shuffle=True)
test_tab_loader = DataLoader(test_tab_ds, batch_size=32, shuffle=False)

mlp_model = DemandMLP(input_size=len(NUMERIC_FEATURES))
mlp_model = train_one_model(mlp_model, train_tab_loader, test_tab_loader,
                             epochs=40, lr=1e-3, writer=writer, tag='MLP', device=device)
mlp_metrics = evaluate(mlp_model, test_tab_loader, device, predict_residual=False)
mlp_naive = naive_baseline_metrics(test_tab_loader, predict_residual=False)
print({k: v for k, v in mlp_metrics.items() if k not in ('y_true', 'y_pred')})
print('Naive (predict = roll_mean_7):', mlp_naive)

writer.close()

## 6. Evaluation

In [ ]:
# Re-assert the inline backend: scripts/*.py switch matplotlib to the headless
# 'Agg' backend when run standalone, and a stale kernel may still be on it.
%matplotlib inline

y_true = lstm_metrics['y_true'][:150]
y_pred = lstm_metrics['y_pred'][:150]

fig, ax = plt.subplots(figsize=(9,4))
ax.plot(range(len(y_true)), y_true, label='Actual', color='#333333')
ax.plot(range(len(y_pred)), y_pred, label='Predicted (LSTM)', color='#C44E52', alpha=0.8)
ax.set_title('Actual vs Predicted units_sold — LSTM (test set, first 150 rows)')
ax.legend()
plt.show()

print(f"LSTM   MAE={lstm_metrics['mae']:.2f} RMSE={lstm_metrics['rmse']:.2f} "
      f"MAPE={lstm_metrics['mape']:.2f}% R2={lstm_metrics['r2']:.3f}")
print(f"MLP    MAE={mlp_metrics['mae']:.2f} RMSE={mlp_metrics['rmse']:.2f} "
      f"MAPE={mlp_metrics['mape']:.2f}% R2={mlp_metrics['r2']:.3f}")
print(f"Naive  MAE={mlp_naive['mae']:.2f} RMSE={mlp_naive['rmse']:.2f} "
      f"MAPE={mlp_naive['mape']:.2f}% R2={mlp_naive['r2']:.3f}   (predict = roll_mean_7, sanity floor both models should beat)")

## 7. Insights & Limitations

_Fill in after running Section 5 with your own results:_
- Which model performed better, and by how much?
- Did either model clear the target thresholds (MAPE ≤ 12%, R² ≥ 0.85)?
- **Sparse sampling:** each SKU has ~99–149 irregular observations over 2 years, so lag/rolling windows are observation-based, not calendar-based.
- **Cold-start limitation:** a new SKU with fewer than `SEQUENCE_LENGTH` (14) historical observations cannot be forecast by the LSTM until enough history accumulates.
- **Ethical consideration:** over-reliance on automated reorder suggestions can itself cause supply chain disruptions if the model fails silently on a demand shock outside its training distribution.